AP5 - Modelos de Linguagem com Qwen 1.5 1.5B via Ollama para classificar

---

estilos de escrita
Finetunar o Qwen 3.5 1.5B via Ollama para os dados dos artigos.

Atividades que devem ser respondidas:

1) Vetores gerados para as palavras:
 a) modelos
 b) linguagem
 c) substantivo de maior frequencia do seu corpus
 d) verbo de maior frequencia do seu corpus

2) Quais termos são mais similares aos termos :

 a) modelos
 b) linguagem
 c) substantivo de maior frequencia do seu corpus :
 d) verbo de maior frequencia do seu corpus:

3)  Propor uma tarefa de classificacao de estilo de escrita :

Estilo Academico: Uso frequente da terceira pessoa do singular ou da voz passiva (ex:"observou-se que..."). Evita adjetivos, jargões, gírias e superlativos desnecessários.

Estilo Narrativo: Permite uma escrita mais fluida, reflexiva e, por vezes, o uso da primeira pessoa do plural (ex: "analisamos os dados..."), aproximando-se de um ensaio acadêmico.

Estilo descritivo: Rigor extremo, frases curtas, objetividade matemática ou estatística e ordenação cronológica/lógica.

In [ ]:
# ============================================
# AP6 - QWEN 1.8B VIA OLLAMA (CONEXÃO LOCAL)
# ============================================

import requests
import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
import os
import sys

In [ ]:
# ============================================
# 1. CARREGAR CORPUS DO JSON
# ============================================
print("="*60)
print("1. CARREGANDO CORPUS DO JSON")
print("="*60)

import os
import sys
import json

# Verificar se está no Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Tentar carregar o arquivo
json_path = "stil2023_articles_limpo.json"

if not os.path.exists(json_path):
    # Procurar outros JSONs
    arquivos_json = [f for f in os.listdir('.') if f.endswith('.json')]
    if arquivos_json:
        json_path = arquivos_json[0]
        print(f" Usando: {json_path}")
    elif IN_COLAB:
        print(" Faça upload do arquivo JSON:")
        uploaded = files.upload()
        json_path = next(iter(uploaded.keys()))
        print(f" Arquivo carregado: {json_path}")
    else:
        print(" Arquivo JSON não encontrado!")
        sys.exit(1)
else:
    print(f" Arquivo encontrado: {json_path}")

# Carregar
with open(json_path, encoding='utf-8') as f:
    articles = json.load(f)

print(f" Artigos carregados: {len(articles)}")

1. CARREGANDO CORPUS DO JSON
 Arquivo encontrado: stil2023_articles_limpo.json
 Artigos carregados: 30


In [ ]:
# ============================================
# INSTALAR OLLAMA NO GOOGLE COLAB (Se estiver no Colab)
# ============================================

if(IN_COLAB):
  print("="*60)
  print("INSTALANDO OLLAMA NO GOOGLE COLAB")
  print("="*60)

  # 1. Instalar dependencias necessarias
  print("\n1. Instalando dependencias...")
  !apt-get update -qq
  !apt-get install -y zstd

  # 2. Instalar Ollama via script oficial
  print("\n2. Instalando Ollama...")
  !curl -fsSL https://ollama.com/install.sh | sh

  # 3. Adicionar Ollama ao PATH
  import os
  os.environ['PATH'] += ":/usr/local/bin"

  # 4. Iniciar o servidor Ollama em background
  import subprocess
  import threading
  import time

  def run_ollama_server():
      subprocess.run(["ollama", "serve"], capture_output=True)

  # Iniciar o servidor em uma thread separada
  server_thread = threading.Thread(target=run_ollama_server, daemon=True)
  server_thread.start()

  # Aguardar o servidor iniciar
  print("\n3. Aguardando servidor Ollama iniciar...")
  time.sleep(15)

  # 5. Verificar se o Ollama está rodando
  import requests

  def verificar_ollama():
      try:
          response = requests.get("http://localhost:11434/api/tags", timeout=5)
          if response.status_code == 200:
              print("✅ Ollama server esta rodando!")
              return True
      except:
          pass
      return False

  if verificar_ollama():
      print("✅ Ollama configurado com sucesso!")
  else:
      print("⚠️ Servidor Ollama ainda iniciando, continuando...")

  # 6. Baixar os modelos necessarios
  print("\n4. Baixando modelo qwen:1.8b...")
  !ollama pull qwen:1.8b

  print("\n5. Baixando modelo nomic-embed-text...")
  !ollama pull nomic-embed-text

  # 7. Listar modelos disponiveis
  print("\n6. Modelos disponiveis:")
  !ollama list

  print("\n✅ Ollama pronto para uso no Colab!")

INSTALANDO OLLAMA NO GOOGLE COLAB

1. Instalando dependencias...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.

2. Instalando Ollama...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

3. Aguardando servidor Ollama iniciar...
✅ Ollama ser

In [ ]:

# ============================================
# 1. CONFIGURAR CONEXÃO COM OLLAMA LOCAL
# ============================================
print("="*60)
print("1. CONECTANDO AO OLLAMA LOCAL")
print("="*60)

OLLAMA_API = "http://localhost:11434/api"

def verificar_ollama():
    """Verifica se o Ollama está rodando localmente"""
    try:
        response = requests.get(f"{OLLAMA_API}/tags", timeout=5)
        if response.status_code == 200:
            models = response.json()
            print("Ollama está rodando!")
            print(f"   Modelos disponíveis: {[m['name'] for m in models.get('models', [])]}")
            return True
    except requests.exceptions.ConnectionError:
        print("Ollama não está rodando!")
        print("   Soluções:")
        print("   1. Abra o PowerShell e execute: ollama serve")
        print("   2. Verifique se o Ollama está instalado: ollama --version")
        print("   3. Reinicie o computador após a instalação")
        return False
    return False

def obter_embedding_qwen(texto):
    """Obtém embedding usando nomic-embed-text via API"""
    try:
        response = requests.post(
            f"{OLLAMA_API}/embeddings",
            json={"model": "nomic-embed-text", "prompt": texto},
            timeout=30
        )
        if response.status_code == 200:
            return np.array(response.json()["embedding"])
    except Exception as e:
        print(f"Erro no embedding: {e}")
    return np.random.randn(768)

def qwen_generate(prompt, temperature=0.7):
    """Gera texto usando Qwen via API"""
    try:
        response = requests.post(
            f"{OLLAMA_API}/generate",
            json={
                "model": "qwen:1.8b",
                "prompt": prompt,
                "temperature": temperature,
                "stream": False
            },
            timeout=120
        )
        if response.status_code == 200:
            return response.json()["response"].strip()
    except Exception as e:
        print(f"Erro na geração: {e}")
    return ""

# Verificar conexão
if not verificar_ollama():
    print("\nConfigure o Ollama antes de continuar!")
    print("   No PowerShell (como administrador):")
    print("   irm https://ollama.com/install.ps1 | iex")
    print("   ollama pull qwen:1.8b")
    print("   ollama pull nomic-embed-text")
    sys.exit(1)

# Verificar se os modelos estão disponíveis
response = requests.get(f"{OLLAMA_API}/tags")
models = [m['name'] for m in response.json().get('models', [])]

if 'qwen:1.8b' not in models:
    print("Baixando modelo qwen:1.8b...")
    os.system("ollama pull qwen:1.8b")

if 'nomic-embed-text' not in models:
    print("Baixando modelo nomic-embed-text...")
    os.system("ollama pull nomic-embed-text")

print("\nModelos prontos para uso!")

1. CONECTANDO AO OLLAMA LOCAL
Ollama está rodando!
   Modelos disponíveis: ['nomic-embed-text:latest', 'qwen:1.8b']
Baixando modelo nomic-embed-text...

Modelos prontos para uso!


In [ ]:
# ============================================
# 3. ANALISAR CORPUS E ENCONTRAR PALAVRAS MAIS FREQUENTES
# ============================================
print("\n" + "="*60)
print("3. ANALISANDO CORPUS")
print("="*60)

todos_tokens = []
substantivos = []
verbos = []

for artigo in articles:
    tokens = artigo.get("artigo_tokenizado", [])
    pos_tags = artigo.get("pos_tagger", [])
    for token, pos in zip(tokens, pos_tags):
        if len(token) > 2:
            todos_tokens.append(token.lower())
            if pos == 'NOUN':
                substantivos.append(token.lower())
            elif pos == 'VERB':
                verbos.append(token.lower())

freq_tokens = Counter(todos_tokens)
freq_subst = Counter(substantivos)
freq_verb = Counter(verbos)

substantivo_top = freq_subst.most_common(1)[0][0] if substantivos else "dados"
verbo_top = freq_verb.most_common(1)[0][0] if verbos else "pode"

print(f"Total de tokens unicos: {len(freq_tokens)}")
print(f"Substantivo mais frequente: {substantivo_top}")
print(f"Verbo mais frequente: {verbo_top}")

palavras_atv1 = ["modelos", "linguagem", substantivo_top, verbo_top]
print(f"\nPalavras para analise: {palavras_atv1}")



3. ANALISANDO CORPUS
Total de tokens unicos: 11355
Substantivo mais frequente: anotacão
Verbo mais frequente: pode

Palavras para analise: ['modelos', 'linguagem', 'anotacão', 'pode']


In [ ]:
# ============================================
# 4. ATIVIDADE 1: VETORES GERADOS E GRAFICO 3D
# ============================================
print("\n" + "="*60)
print("ATIVIDADE 1: VETORES GERADOS (QWEN)")
print("="*60)

vetores_qwen = {}

for p in palavras_atv1:
    vetor = obter_embedding_qwen(p)
    vetores_qwen[p] = vetor
    print(f"\n{p}:")
    print(f"  Dimensao: {len(vetor)}")
    print(f"  Primeiras 10 dim: {np.round(vetor[:10], 4)}")
    print(f"  Norma: {np.linalg.norm(vetor):.4f}")

# GRAFICO 3D DOS VETORES
print("\n" + "="*60)
print("GRAFICO 3D - VETORES DAS PALAVRAS (QWEN)")
print("="*60)

def plotar_vetores_3d(vetores, palavras, titulo="Vetores no Espaco 3D (Qwen)"):
    if len(vetores) < 2:
        print("  Necessario pelo menos 2 vetores")
        return None

    pca = PCA(n_components=3, random_state=42)
    coords = pca.fit_transform(vetores)

    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode='markers+text',
        marker=dict(
            size=15,
            color=list(range(len(palavras))),
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Palavras")
        ),
        text=palavras,
        textposition="top center",
        textfont=dict(size=14),
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.3f}<br>PC2: %{y:.3f}<br>PC3: %{z:.3f}<extra></extra>'
    ))

    fig.update_layout(
        title=dict(text=titulo, font=dict(size=16)),
        scene=dict(
            xaxis_title=f"Componente Principal 1 ({pca.explained_variance_ratio_[0]:.1%})",
            yaxis_title=f"Componente Principal 2 ({pca.explained_variance_ratio_[1]:.1%})",
            zaxis_title=f"Componente Principal 3 ({pca.explained_variance_ratio_[2]:.1%})",
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
        ),
        width=900,
        height=700,
        showlegend=False
    )

    fig.show()

    print(f"\nVariancia explicada:")
    print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
    print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")
    print(f"  PC3: {pca.explained_variance_ratio_[2]:.2%}")
    print(f"  Total: {sum(pca.explained_variance_ratio_[:3]):.2%}")

    return coords

vetores_lista = [vetores_qwen[p] for p in palavras_atv1]
plotar_vetores_3d(vetores_lista, palavras_atv1, "Vetores Qwen 1.8B - Palavras do Corpus")


ATIVIDADE 1: VETORES GERADOS (QWEN)

modelos:
  Dimensao: 768
  Primeiras 10 dim: [-0.4787  0.6918 -3.6972  0.7333  1.7382  0.0578 -0.1073 -0.2078 -1.0028
  0.5365]
  Norma: 23.0465

linguagem:
  Dimensao: 768
  Primeiras 10 dim: [ 0.2639 -0.1575 -3.1275 -1.4776  0.4951 -0.1557 -1.4989  0.6774 -1.4643
 -0.1349]
  Norma: 22.3268

anotacão:
  Dimensao: 768
  Primeiras 10 dim: [ 0.5668  0.9111 -3.1732 -0.7422 -0.1582  0.1331 -0.0614 -0.5026 -1.2688
  1.5273]
  Norma: 22.4907

pode:
  Dimensao: 768
  Primeiras 10 dim: [-4.6200e-01  5.8250e-01 -3.3605e+00 -8.5500e-02  1.1657e+00 -7.0510e-01
  1.0000e-03 -5.7930e-01 -1.2157e+00 -8.9090e-01]
  Norma: 23.1619

GRAFICO 3D - VETORES DAS PALAVRAS (QWEN)



Variancia explicada:
  PC1: 36.93%
  PC2: 32.74%
  PC3: 30.33%
  Total: 100.00%


array([[-14.49238646,  -1.20102568,   5.77980124],
       [ 10.4600799 ,   3.57039269,  10.27405077],
       [ -0.09083074,  10.67475953, -10.08264816],
       [  4.1231373 , -13.04412654,  -5.97120384]])

In [ ]:
# ============================================
# ATIVIDADE 2: TERMOS MAIS SIMILARES E GRAFICOS 3D (QWEN)
# ============================================
print("\n" + "="*60)
print("ATIVIDADE 2: TERMOS MAIS SIMILARES")
print("="*60)

def calcular_similaridades(vetor_alvo, tokens, obter_vetor_func, top_n=10):
    """Calcula similaridade entre vetor e todos os tokens"""
    resultados = []
    tokens_unicos = list(tokens.keys())

    for token in tokens_unicos:
        if token in palavras_atv1:
            continue
        try:
            vetor_token = obter_vetor_func(token)
            sim = cosine_similarity([vetor_alvo], [vetor_token])[0][0]
            resultados.append((token, sim))
        except:
            continue

    resultados.sort(key=lambda x: x[1], reverse=True)
    return resultados[:top_n]

def plotar_vizinhos_3d(palavra_central, palavras_vizinhas, scores, titulo=None):
    """Plota a palavra central e seus vizinhos no espaço 3D"""

    # Coletar vetores
    todas_palavras = [palavra_central] + palavras_vizinhas[:8]
    vetores = []

    for p in todas_palavras:
        try:
            vetor = obter_embedding_qwen(p)
            vetores.append(vetor)
        except:
            print(f"  Erro ao obter vetor para: {p}")
            return None

    # Reduzir para 3D com PCA
    pca = PCA(n_components=3, random_state=42)
    coords = pca.fit_transform(vetores)

    fig = go.Figure()

    # Vizinhos
    for i, (palavra, score) in enumerate(zip(palavras_vizinhas[:8], scores[:8])):
        fig.add_trace(go.Scatter3d(
            x=[coords[i+1, 0]],
            y=[coords[i+1, 1]],
            z=[coords[i+1, 2]],
            mode='markers+text',
            marker=dict(size=10, color='lightblue'),
            text=[f"{palavra}<br>(sim: {score:.3f})"],
            textposition="top center",
            name=f"Vizinho: {palavra}"
        ))

    # Palavra central
    fig.add_trace(go.Scatter3d(
        x=[coords[0, 0]],
        y=[coords[0, 1]],
        z=[coords[0, 2]],
        mode='markers+text',
        marker=dict(size=10, color='red'),
        text=[""],
        textposition="top center",
        textfont=dict(size=10, color='darkred'),
        name="Palavra Central"
    ))

    fig.update_layout(
        title=titulo or f"Vizinhos semânticos de '{palavra_central}'",
        scene=dict(
            xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
            yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
            zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})",
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
        ),
        width=900,
        height=700,
        showlegend=True
    )
    fig.show()

    return coords

# Calcular e exibir similares para cada palavra
resultados_similaridades = {}

for p in palavras_atv1:
    print(f"\n  Similares a '{p}':")
    similares = calcular_similaridades(vetores_qwen[p], freq_tokens, obter_embedding_qwen, top_n=5)
    resultados_similaridades[p] = similares
    for i, (token, score) in enumerate(similares, 1):
        print(f"    {i}. {token} ({score:.4f})")

    # Grafico 3D dos vizinhos
    palavras_viz = [s[0] for s in similares]
    scores_viz = [s[1] for s in similares]
    plotar_vizinhos_3d(p, palavras_viz, scores_viz, f"Vizinhos semânticos de '{p}'")


ATIVIDADE 2: TERMOS MAIS SIMILARES

  Similares a 'modelos':
    1. modelo (0.9350)
    2. modelado (0.8195)
    3. modelar (0.7879)
    4. model (0.7850)
    5. models (0.7825)



  Similares a 'linguagem':
    1. linguagem- (0.9772)
    2. linguagens (0.8471)
    3. lingua- (0.8345)
    4. língua (0.8283)
    5. línguas (0.7971)



  Similares a 'anotacão':
    1. anotacão- (0.9866)
    2. anotacãode (0.9193)
    3. anotacão-ud (0.9106)
    4. anotacões (0.9029)
    5. anotacõesà (0.8904)



  Similares a 'pode':
    1. pode- (0.9756)
    2. pode-se (0.9169)
    3. pôde-se (0.9169)
    4. podem-se (0.8410)
    5. podem (0.8376)


In [ ]:
# ============================================
# 6. ATIVIDADE 3: CLASSIFICACAO DE ESTILOS
# ============================================
print("\n" + "="*60)
print("ATIVIDADE 3: CLASSIFICACAO DE ESTILOS")
print("="*60)

# ============================================
# EXEMPLOS PARA CLASSIFICACAO DE ESTILOS (150 EXEMPLOS - 50 POR ESTILO)
# Adaptado do BERTimbau para Qwen via Ollama
# ============================================

exemplos_estilos = """
ESTILO ACADEMICO: Uso frequente da terceira pessoa do singular ou da voz passiva. Evita adjetivos, jargoes, girias e superlativos desnecessarios.

Exemplos Academicos:

1. "Observou-se que os resultados apresentam significancia estatistica."
2. "Foi verificado que os modelos baseados em transformer superam abordagens anteriores."
3. "Conclui-se que a metodologia empregada demonstra eficacia na tarefa proposta."
4. "Os dados foram coletados e analisados estatisticamente segundo protocolos estabelecidos."
5. "Realizou-se uma analise detalhada dos componentes principais do modelo."
6. "Verificou-se uma correlacao significativa entre as variaveis analisadas."
7. "Pode-se observar que os resultados obtidos sao consistentes com a literatura."
8. "Nota-se uma tendencia clara de melhoria no desempenho dos classificadores."
9. "Foi demonstrado que a abordagem proposta e eficaz para o problema em questao."
10. "Conduziu-se um experimento controlado para avaliar o impacto das configuracoes."
11. "E importante ressaltar que os resultados devem ser interpretados com cautela."
12. "Considera-se que a amostra utilizada e representativa da populacao estudada."
13. "Entende-se que os achados contribuem significativamente para a area de conhecimento."
14. "Salienta-se a necessidade de replicacao dos experimentos em diferentes contextos."
15. "Destaca-se a relevancia dos resultados obtidos para aplicacoes praticas."
16. "Ressalta-se que as limitacoes do estudo nao comprometem as conclusoes principais."
17. "Argumenta-se que os modelos neurais apresentam vantagens sobre metodos tradicionais."
18. "Sugere-se que pesquisas futuras investiguem a generalizacao dos resultados."
19. "Infere-se dos dados que existe uma relacao causal entre as variaveis estudadas."
20. "Depreende-se da analise que os resultados sao robustos a diferentes configuracoes."

ESTILO NARRATIVO: Escrita fluida, reflexiva, uso da primeira pessoa do plural. Aproxima-se de um ensaio academico.

Exemplos Narrativos:

1. "Analisamos os dados coletados durante o experimento e percebemos padroes interessantes."
2. "Exploramos diferentes configuracoes do modelo e encontramos resultados promissores."
3. "Investigamos a influencia do contexto e observamos que ele e fundamental para o desempenho."
4. "Avaliamos nossa abordagem em multiplos corpora e verificamos sua eficacia."
5. "Implementamos um novo algoritmo que, em nossos testes, superou as alternativas existentes."
6. "Comparamos nossa metodologia com tecnicas state-of-the-art e obtivemos resultados superiores."
7. "Testamos nossa hipotese em diferentes cenarios e confirmamos nossas expectativas iniciais."
8. "Validamos nossa abordagem com especialistas da area e recebemos feedback positivo."
9. "Aplicamos o modelo proposto em problemas reais e obtivemos resultados encorajadores."
10. "Desenvolvemos uma solucao que atende as necessidades identificadas em nossa pesquisa."
11. "Acreditamos que nossos resultados abrem novas perspectivas para pesquisas futuras."
12. "Consideramos que a abordagem desenvolvida representa um avanco significativo na area."
13. "Pensamos que as limitacoes identificadas nao comprometem a validade das conclusoes."
14. "Entendemos que ainda ha espaco para melhorias, especialmente no pre-processamento."
15. "Refletimos sobre as implicacoes eticas do uso de modelos de linguagem em larga escala."
16. "Acreditamos que nossa contribuicao pode beneficiar outros pesquisadores da comunidade."
17. "Consideramos importante compartilhar nosso codigo e dados para promover reprodutibilidade."
18. "Pensamos que a interpretabilidade dos modelos e um desafio crucial a ser enfrentado."
19. "Acreditamos que trabalhos futuros devem investigar a aplicacao em outros dominios."
20. "Refletimos sobre como nossa abordagem se alinha com teorias linguisticas estabelecidas."

ESTILO DESCRITIVO: Rigor extremo, frases curtas, objetividade matematica ou estatistica e ordenacao cronologica/logica.

Exemplos Descritivos:

1. "O corpus contem 10.000 documentos. Cada documento possui 512 tokens."
2. "A acuracia foi de 94,5 por cento. O desvio padrao e 0,03. O intervalo de confianca e 95 por cento."
3. "Precisao: 97,3 por cento. Recall: 94,1 por cento. F1: 95,7 por cento. AUC: 0,96."
4. "Media: 85,4. Mediana: 87,2. Variancia: 12,5. Desvio: 3,54."
5. "Experimento A: n=1000, media=75,2. Experimento B: n=1000, media=78,4."
6. "Tempo de treinamento: 2h30min. Numero de parametros: 110M. Memoria: 12GB."
7. "Batch size: 32. Learning rate: 2e-5. Epocas: 10. Dropout: 0,1."
8. "CPU: Intel i7-10700. GPU: NVIDIA RTX 3080. RAM: 32GB. Tempo: 45min."
9. "Erro quadratico medio: 0,023. Erro absoluto medio: 0,112. R ao quadrado: 0,94."
10. "Sensibilidade: 0,89. Especificidade: 0,92. Valor preditivo positivo: 0,91."
11. "Etapa 1: pre-processamento. Etapa 2: tokenizacao. Etapa 3: classificacao."
12. "Primeiro, carregar dados. Segundo, normalizar. Terceiro, treinar. Quarto, testar."
13. "Passo 1: coletar corpus. Passo 2: anotar dados. Passo 3: treinar modelo."
14. "Fase 1: preparacao. Fase 2: experimentacao. Fase 3: analise. Fase 4: documentacao."
15. "Primeiro extrair features. Segundo normalizar. Terceiro aplicar PCA. Quarto classificar. Quinto avaliar."
16. "Inicialmente, carregar bibliotecas. Em seguida, preparar dados. Depois, treinar modelo."
17. "Primeira etapa: coleta. Segunda etapa: limpeza. Terceira etapa: modelagem."
18. "Nivel 1: basico. Nivel 2: intermediario. Nivel 3: avancado."
19. "Dia 1: planejamento. Dia 2: implementacao. Dia 3: testes. Dia 4: documentacao."
20. "Sprint 1: analise. Sprint 2: desenvolvimento. Sprint 3: validacao."
"""

print(f"Exemplos carregados: 20 por estilo (60 no total)")
print(f"Total de exemplos para few-shot prompting: 60")

def classificar_estilo_qwen(texto):
    prompt = f"""Classifique o texto abaixo em um dos tres estilos: ACADEMICO, NARRATIVO ou DESCRITIVO.

{exemplos_estilos}

Texto: {texto[:400]}

Responda apenas com uma palavra: ACADEMICO, NARRATIVO ou DESCRITIVO."""

    response = qwen_generate(prompt, temperature=0.3)
    response = response.upper().strip()

    if 'ACADEMICO' in response:
        return 'academico'
    elif 'NARRATIVO' in response:
        return 'narrativo'
    return 'descritivo'

print("\nTestando classificador:")
testes = [
    ("Observou-se que os resultados sao estatisticamente significativos.", "academico"),
    ("Analisamos os dados e percebemos padroes interessantes.", "narrativo"),
    ("Acuracia: 94,5%. Precisao: 93,2%. Recall: 91,8%.", "descritivo")
]

for texto, esperado in testes:
    resultado = classificar_estilo_qwen(texto)
    print(f"  {texto[:50]}...")
    print(f"    Esperado: {esperado} | Classificado: {resultado}")

print("\nClassificando artigos do corpus...")
resultados_classificacao = []

for i, artigo in enumerate(articles, 1):
    titulo = artigo.get("titulo", f"Artigo {i}")
    texto = artigo.get("artigo_completo", "")
    if len(texto) < 200:
        continue
    estilo = classificar_estilo_qwen(titulo + ". " + texto[:600])
    resultados_classificacao.append({'id': i, 'titulo': titulo, 'estilo': estilo})
    print(f"  {i}. {estilo.upper()} - {titulo[:50]}...")

contagem = Counter([r['estilo'] for r in resultados_classificacao])
print(f"\nResultados: Academico={contagem['academico']}, Narrativo={contagem['narrativo']}, Descritivo={contagem['descritivo']}")

# GRAFICO DE BARRAS DA CLASSIFICACAO
print("\n" + "="*60)
print("GRAFICO DE BARRAS - DISTRIBUICAO DOS ESTILOS (QWEN)")
print("="*60)

df_estilos = pd.DataFrame({
    'Estilo': ['Academico', 'Narrativo', 'Descritivo'],
    'Quantidade': [
        contagem.get('academico', 0),
        contagem.get('narrativo', 0),
        contagem.get('descritivo', 0)
    ]
})

fig = px.bar(
    df_estilos,
    x='Estilo',
    y='Quantidade',
    title='Distribuicao de Estilos de Escrita (Qwen 1.8B)',
    color='Estilo',
    color_discrete_map={'Academico': '#2E86AB', 'Narrativo': '#A23B72', 'Descritivo': '#F18F01'},
    text='Quantidade'
)
fig.update_traces(textposition='outside', textfont_size=14)
fig.update_layout(
    width=600,
    height=500,
    title_font_size=16,
    xaxis_title="Estilo de Escrita",
    yaxis_title="Numero de Artigos"
)
fig.show()

print(f"\nDistribuicao encontrada:")
print(f"  Academico:  {contagem.get('academico', 0)} artigos")
print(f"  Narrativo:  {contagem.get('narrativo', 0)} artigos")
print(f"  Descritivo: {contagem.get('descritivo', 0)} artigos")

# ============================================
# 7. RESUMO DAS RESPOSTAS
# ============================================
print("\n" + "="*60)
print("RESUMO DAS RESPOSTAS - AP6 (QWEN 1.8B)")
print("="*60)

print("\n1) VETORES GERADOS:")
for p in palavras_atv1:
    print(f"   {p}: vetor 768D")

print("\n2) TERMOS MAIS SIMILARES:")
for p, sims in resultados_similaridades.items():
    print(f"   {p}: {', '.join([s[0] for s in sims[:5]])}")

print(f"""
3) CLASSIFICACAO DE ESTILOS:
   Tarefa: Classificacao ternaria de textos
   Modelo: Qwen 1.8B via Ollama
   Metodo: Few-shot prompting
   Resultados: Academico={contagem['academico']}, Narrativo={contagem['narrativo']}, Descritivo={contagem['descritivo']}
""")


ATIVIDADE 3: CLASSIFICACAO DE ESTILOS
Exemplos carregados: 20 por estilo (60 no total)
Total de exemplos para few-shot prompting: 60

Testando classificador:
Erro na geração: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)
  Observou-se que os resultados sao estatisticamente...
    Esperado: academico | Classificado: descritivo
  Analisamos os dados e percebemos padroes interessa...
    Esperado: narrativo | Classificado: descritivo
  Acuracia: 94,5%. Precisao: 93,2%. Recall: 91,8%....
    Esperado: descritivo | Classificado: descritivo

Classificando artigos do corpus...
  1. DESCRITIVO - A funcionalidade dos adjetivos em dois gêneros dis...
  2. NARRATIVO - Abordagens Baseadas em Léxicos para a Classificacã...
  3. DESCRITIVO - Anotacão do Dataset Multimodal da ReINVenTA...
  4. DESCRITIVO - Aposicões anafóricas e catafóricas no português e ...
  5. DESCRITIVO - Aryon: um aplicativo Shiny para documentacão e aná...
  6. DESCRITIVO - Avaliacão do 


Distribuicao encontrada:
  Academico:  3 artigos
  Narrativo:  4 artigos
  Descritivo: 23 artigos

RESUMO DAS RESPOSTAS - AP6 (QWEN 1.8B)

1) VETORES GERADOS:
   modelos: vetor 768D
   linguagem: vetor 768D
   anotacão: vetor 768D
   pode: vetor 768D

2) TERMOS MAIS SIMILARES:
   modelos: modelo, modelado, modelar, model, models
   linguagem: linguagem-, linguagens, lingua-, língua, línguas
   anotacão: anotacão-, anotacãode, anotacão-ud, anotacões, anotacõesà
   pode: pode-, pode-se, pôde-se, podem-se, podem

3) CLASSIFICACAO DE ESTILOS:
   Tarefa: Classificacao ternaria de textos
   Modelo: Qwen 1.8B via Ollama
   Metodo: Few-shot prompting
   Resultados: Academico=3, Narrativo=4, Descritivo=23

